In [ ]:
# Check whether it's Tesla T4
!nvidia-smi

# Tweak model to Google T4: Edit -> Notebook Settings -> T4 GPU

In [ ]:
# pip install necessary items
!pip install ultralytics roboflow -q
import ultralytics, os, roboflow
from ultralytics import YOLO
from roboflow import Roboflow

In [ ]:
# Introduce Roboflow api

rf = Roboflow(api_key="jSEZZq0JHLG1mvUYpHVO")
project = rf.workspace("fyp-ormnr").project("supermarket-empty-shelf-detector")
version = project.version(4)
dataset = version.download("yolov11")

In [ ]:
# # tweak the Roboflow path with both val and valid
# path = 'content/Supermarket-Empty_shelf-Detector--4'
# dataset.location = os.path.join(path, 'yaml') # os.path.join('user', folder, filename)

In [ ]:
# Download YOLO11n and run train the data
model = YOLO("yolo11n.pt")

results = model.train(data=os.path.join(dataset.location, "data.yaml"), epochs=50)

In [ ]:
# Check mAP 50 and mAP50-95

metrics = model.val(data = os.path.join(dataset.location, "data.yaml"))

print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# !ls /content/Supermarket-Empty-Shelf-Detector--4

In [ ]:
# Display the valid images with boxes
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

def validate_images(path, img_count = -1):
  # 1. Grab all images from your validation folder
  val_images = glob.glob(path)

  # 2. Run inference on a few validation images
  results = model.predict(source=val_images[:img_count], conf=0.25)

  # 3. Display the images with bounding boxes
  for r in results:
      # Plot returns a BGR numpy array with boxes drawn
      im_bgr = r.plot()

      # Convert BGR to RGB for correct colors in Matplotlib/Notebooks
      im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)

      # Show using matplotlib
      plt.figure(figsize=(10, 10))
      plt.imshow(im_rgb)
      plt.axis("off")
      plt.show()

validate_images("/content/Supermarket-Empty-Shelf-Detector--4/valid/images/*.jpg", 3)

In [ ]:
# Check empty area/ overall area rate

def empty_area_rate(path, img_count = -1):
  # 1. Grab all images from your validation folder
  val_images = glob.glob(path)

  # 2. Run inference on a few validation images
  results = model.predict(source=val_images[:img_count], conf=0.25)

  # 3. Display the images with bounding boxes
  img = 1
  for r in results:
    h_img, w_img = r.orig_shape
    ttl_boxes_area = 0
    for _, _, w_box, h_box in r.boxes.xywh:
      ttl_boxes_area += h_box*w_box
    rate = round(float(100* ttl_boxes_area / (h_img*w_img)), 2)
    print(f"Image {img} has {rate}% of total area of the image is empty.")
    img += 1
empty_area_rate("/content/Supermarket-Empty-Shelf-Detector--4/valid/images/*.jpg", 3)

In [ ]:
# Time to test my own images!

# Create a folder and upload my own images to the folder
from google.colab import files
import shutil

# Make a new folder for my images
path = '/content/Supermarket-Empty-Shelf-Detector--4/my_images'
os.makedirs(path, exist_ok = True)

# This opens the upload prompt
uploaded = files.upload()

# This automatically moves your uploaded images into the folder you made
for filename in uploaded.keys():
    shutil.move(filename, path)


In [ ]:
# !ls /content/Supermarket-Empty-Shelf-Detector--4/my_images

In [ ]:
# Convert HEIC to jpg
!pip install pillow-heif -q

from PIL import Image
import pillow_heif

# Define your folders
input_folder = '/content/Supermarket-Empty-Shelf-Detector--4/my_images'
output_folder = '/content/Supermarket-Empty-Shelf-Detector--4/my_images'

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Loop through all files in the input folder
for filename in os.listdir(input_folder):
    if filename.lower().endswith('.heic'):
        heic_path = os.path.join(input_folder, filename)

        try:
            # Open HEIC image
            heic_image = Image.open(heic_path)

            # Create JPG filename (replace .heic extension with .jpg)
            base_name = os.path.splitext(filename)[0]
            jpg_filename = f"{base_name}.jpg"
            jpg_path = os.path.join(output_folder, jpg_filename)

            # Convert and save as JPG
            heic_image.convert('RGB').save(jpg_path, 'JPEG', quality=90)
            print(f"Converted: {filename} -> {jpg_filename}")

        except Exception as e:
            print(f"Failed to convert {filename}: {e}")

print("Batch conversion completed!")

In [ ]:
# Test it on my own data!

validate_images("/content/Supermarket-Empty-Shelf-Detector--4/my_images/*.jpg")

In [ ]:
empty_area_rate("/content/Supermarket-Empty-Shelf-Detector--4/my_images/*.jpg")

In [ ]:
!ls /content/Supermarket-Empty-Shelf-Detector--4/my_images